In [4]:
### Implementation of the ResNet Basic Block

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

## GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")


## ResNet 기본 블록 정의
class BasicBlock(nn.Module):
  expansion = 1

  def __init__(self, in_channels, out_channels, stride=1, downsample=None):
    super(BasicBlock, self).__init__() # nn.Module 상속 → 메서드/함수/속성 사용
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                           stride=stride, padding=1, bias=False) # 합성곱층 1
    self.bn1 = nn.BatchNorm2d(out_channels) # 배치 정규화층 1
    self.relu = nn.ReLU(inplace=True)
    # inplace → 새로운 텐서를 생성 X. 입력 텐서 값을 직접 덮어써서 수정
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                           padding=1, bias=False) #합성곱층 2
    self.bn2 = nn.BatchNorm2d(out_channels) # 배치 정규화층 2
    self.downsample = downsample
    self.stride = stride

  def forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)


    out = self.conv2(out)
    out = self.bn2(out)

    '''
    downsample 역할
    → 입력 데이터와 잔차 블록의 출력 데이터 크기(차원)를 일치!
    '''
    if self.downsample is not None:
      identity = self.downsample(x)

    out += identity # Skip Connection(잔차 연결)
    out = self.relu(out)

    return out # End of Class BasicBlock(nn.Module)


## 테스트 코드
def test_basic_block():
  # 샘플 입력 생성
  x = torch.randn(1, 64, 32, 32).to(device) # (N, C, H, W)

  # BasicBlock 생성(다운샘플링 X)
  block1 = BasicBlock(64, 64).to(device)
  output1 = block1(x)
  print(f"동일 채널 블록 출력 크기: {output1.shape}")

  # downsample 구성 시: 입력 채널(64)과 출력 채널(128)이 다르므로,
  # 1x1 Conv를 통해 채널 수를 128로 늘리고 가로세로 크기를 조절한다.
  # 이후 BatchNorm을 적용한다.
  downsample = nn.Sequential(
      nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False),
      nn.BatchNorm2d(128)
  ).to(device)

  block2 = BasicBlock(64, 128, stride=2, downsample=downsample).to(device)
  output2 = block2(x)
  print(f"다운샘플링 블록 출력 크기: {output2.shape}")


test_basic_block()

사용 중인 디바이스: cuda
동일 채널 블록 출력 크기: torch.Size([1, 64, 32, 32])
다운샘플링 블록 출력 크기: torch.Size([1, 128, 16, 16])
